# Đổi keywords trong cell dưới đây r cào

In [5]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

MAX_PLACES_PER_KEYWORD = 40

driver = webdriver.Chrome()
driver.get("https://www.google.com/maps")

wait = WebDriverWait(driver, 20)

search_box = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, 'input[name="q"]'))
)

# 
keywords = [
    # "quán cơm quận 6",
    # "quán phở quận 6",
    # "quán ăn thái quận 6",
    # "quán ăn nhật quận 6",
    # "quán ăn hàn quận 6",

    # "quán cơm quận 7",
    # "quán phở quận 7",
    # "quán ăn thái quận 7",
    # "quán ăn nhật quận 7",
    # "quán ăn hàn quận 7",
    
    "quán cơm quận 1",
    "quán phở quận 1",
    "quán ăn thái quận 1",
    "quán ăn nhật quận 1",
    "quán ăn hàn quận 1",

    # "quán cơm quận 10",
    # "quán phở quận 10",
    # "quán ăn thái quận 10",
    # "quán ăn nhật quận 10",
    # "quán ăn hàn quận 10",
]


def search(keyword):
    while True:
        try:
            search_box = wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'input[name="q"]'))
            )

            search_box.clear()
            time.sleep(1)

            search_box.send_keys(keyword)
            search_box.send_keys(Keys.ENTER)

            # chờ list load
            wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="feed"]'))
            )

            time.sleep(2)
            break

        except:
            print("Search retry...")
            time.sleep(1)

def collect_urls(max_urls=None):
    urls = set()

    last_count = 0
    same_count_times = 0
    scroll_retry = 0

    while len(urls) < max_urls:

        try:
            # 🔥 luôn lấy lại panel (tránh stale)
            panel = wait.until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div[role="feed"]'))
            )

            # 🔥 scroll với retry
            success = False
            for _ in range(3):
                try:
                    driver.execute_script(
                        "arguments[0].scrollTop = arguments[0].scrollHeight",
                        panel
                    )
                    success = True
                    break
                except:
                    time.sleep(1)

            if not success:
                print("Scroll failed → retry loop")
                scroll_retry += 1
                if scroll_retry >= 3:
                    break
                continue

        except:
            print("Panel stale → retry...")
            time.sleep(1)
            continue

        time.sleep(2)

        # 🔥 lấy lại elements mỗi lần (tránh stale)
        try:
            elements = driver.find_elements(By.CSS_SELECTOR, 'a.hfpxzc')
        except:
            print("Elements stale → retry...")
            continue

        for e in elements:
            try:
                url = e.get_attribute("href")

                if not url:
                    continue

                # ✅ chỉ lấy link place
                if "/maps/place/" not in url:
                    continue

                # ❌ lọc sponsor (basic)
                try:
                    parent = e.find_element(By.XPATH, "./ancestor::div[@role='article']")
                    if "Sponsored" in parent.text:
                        continue
                except:
                    pass

                urls.add(url)

            except:
                continue

        print(f"Collected: {len(urls)}")

        # 🔥 check tăng trưởng
        if len(urls) == last_count:
            same_count_times += 1
        else:
            same_count_times = 0

        # ✅ dừng khi không còn load thêm
        if same_count_times >= 3:
            print("No more new data")
            break

        last_count = len(urls)

    return urls
# 🔥 MAIN LOOP
all_urls = set()

for keyword in keywords:
    print(f"\n===== Searching: {keyword} =====")

    search(keyword)

    # 👉 trick: zoom nhẹ để đổi viewport (lấy thêm quán)
    driver.execute_script("document.body.style.zoom='90%'")
    time.sleep(1)

    urls = collect_urls(max_urls=40)

    all_urls.update(urls)

    print("Total unique URLs:", len(all_urls))

driver.quit()

# ghi file
with open("urls_quan_1.txt", "w", encoding="utf-8") as f:
    for url in all_urls:
        f.write(url + "\n")

print("DONE. Total URLs:", len(all_urls))


===== Searching: quán cơm quận 1 =====
Collected: 12
Collected: 19
Collected: 20
Collected: 23
Collected: 30
Collected: 37
Collected: 43
Total unique URLs: 43

===== Searching: quán phở quận 1 =====
Collected: 8
Collected: 13
Collected: 18
Collected: 20
Collected: 23
Collected: 26
Collected: 30
Collected: 34
Collected: 37
Collected: 41
Total unique URLs: 84

===== Searching: quán ăn thái quận 1 =====
Collected: 10
Collected: 15
Collected: 20
Collected: 24
Collected: 28
Collected: 33
Collected: 38
Collected: 41
Total unique URLs: 121

===== Searching: quán ăn nhật quận 1 =====
Collected: 8
Collected: 13
Collected: 18
Collected: 20
Collected: 20
Collected: 28
Collected: 33
Collected: 38
Collected: 40
Total unique URLs: 153

===== Searching: quán ăn hàn quận 1 =====
Collected: 10
Collected: 15
Collected: 20
Collected: 21
Collected: 25
Collected: 30
Collected: 34
Collected: 39
Collected: 43
Total unique URLs: 189
DONE. Total URLs: 189


In [6]:
# --- Bước 1: Gộp tất cả URL ---
files_to_merge = [
    "urls_quan_1.txt",
    "urls_quan_6.txt",
    "urls_quan_7.txt",
    "urls_quan_10.txt"
]

all_urls_merged = set()

for file in files_to_merge:
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            url = line.strip()
            if url:  # bỏ dòng rỗng
                all_urls_merged.add(url)

print(f"Total URLs after merge: {len(all_urls_merged)}")

# --- Bước 2: Load URLs từ ../urls.txt để so sánh ---
with open("../urls.txt", "r", encoding="utf-8") as f:
    urls_quan_5 = set(line.strip() for line in f if line.strip())

# --- Bước 3: Loại bỏ URL trùng ---
urls_final = all_urls_merged - urls_quan_5
print(f"Total URLs after removing duplicates with quận 5: {len(urls_final)}")

# --- Bước 4: Lưu vào file url_ngoai_quan_5.txt ---
with open("url_ngoai_quan_5.txt", "w", encoding="utf-8") as f:
    for url in sorted(urls_final):  # sắp xếp để dễ kiểm tra
        f.write(url + "\n")

print("DONE. File 'url_ngoai_quan_5.txt' đã được cập nhật.")

Total URLs after merge: 600
Total URLs after removing duplicates with quận 5: 530
DONE. File 'url_ngoai_quan_5.txt' đã được cập nhật.
